# 🔎 Clase 01 · Práctica guiada de EDA
Dataset: **viviendas en Lima** (sintético, con problemas puestos a propósito).
Objetivo del futuro modelo: predecir `precio_usd`.

Ejecuta celda por celda (`Shift + Enter`). En las celdas **✏️ Tu turno** escribes tú.
Al final implementa `ejercicio.py` y evalúa con `make evaluar CLASE=...`.

In [ ]:
import sys
from pathlib import Path

# Localiza la raíz del repo, para que el notebook funcione desde cualquier carpeta
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "framework").exists())
sys.path.insert(0, str(RAIZ))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from framework.testing import ruta_dataset  # si el CSV no existe, lo genera

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

## Paso 0 · Carga de datos
En la vida real esto vendría de una consulta SQL o de un CSV que te entrega el ingeniero de datos.

In [ ]:
df = pd.read_csv(ruta_dataset("viviendas_lima.csv"))
df.head()

In [ ]:
df.info()

### ¿Duplicados?
Una fila duplicada infla los conteos y puede filtrar información entre train y test más adelante.

In [ ]:
print("Filas duplicadas:", df.duplicated().sum())
df[df.duplicated(keep=False)].sort_values("id").head(6)

## Paso 1 · Variable objetivo
En este dataset es `precio_usd`, una variable **numérica continua**, así que el problema es de **regresión**.

In [ ]:
OBJETIVO = "precio_usd"
df[OBJETIVO].describe()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df[OBJETIVO], bins=60, ax=ax[0]).set_title("precio_usd")
sns.histplot(np.log10(df[OBJETIVO]), bins=60, ax=ax[1]).set_title("log10(precio_usd)")
plt.tight_layout()

> 💬 El precio tiene una **cola larga a la derecha** (asimetría positiva): muchas viviendas "normales" y unas pocas carísimas.
> Con logaritmo se ve más simétrico. Esto se retoma con outliers (módulo 2).

## Paso 2 · Tipos de variables

In [ ]:
numericas = df.select_dtypes(include="number").columns.drop(["id", OBJETIVO]).tolist()
categoricas = df.select_dtypes(exclude="number").columns.tolist()
print("Numéricas  :", numericas)
print("Categóricas:", categoricas)

> ⚠️ **Ojo:** que una columna sea número no la vuelve "numérica" para el análisis. `habitaciones` es un conteo discreto, y un `id` no tiene ningún significado numérico.
> El tipo de dato (dtype) y el tipo de variable (estadístico) no siempre coinciden.

## Paso 3a · Distribución de las numéricas

In [ ]:
resumen = df[numericas].agg(["mean", "median", "std", "min", "max", "skew"]).T
resumen.round(2)

In [ ]:
df[numericas].hist(bins=40, figsize=(14, 8))
plt.tight_layout()

### ✏️ Tu turno
Mira `resumen` y los histogramas y responde (en una celda markdown):
1. ¿Qué columna tiene un **máximo absurdo**? Pista: compara `max` con la mediana.
2. ¿Hay alguna antigüedad imposible?

In [ ]:
# Tu código aquí (por ejemplo: df.sort_values('area_m2', ascending=False).head())


## Paso 3b · Conteo de las categóricas

In [ ]:
for col in categoricas:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(8))

> 🕵️ ¿Viste algo raro en `cochera`? Aparecen `si`, `SI` y `no `: **texto sucio**. Para pandas son categorías distintas.
> Y `pais` es casi constante: una variable de **baja varianza** que no aporta nada (se ve en el módulo 3).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
df["tipo_inmueble"].value_counts().plot.bar(ax=ax[0], title="tipo_inmueble")
df["distrito"].value_counts().plot.barh(ax=ax[1], title="distrito")
plt.tight_layout()

## Paso 3c · Análisis bivariante: ¿qué variables diferencian el precio?

In [ ]:
por_distrito = (df.groupby("distrito")[OBJETIVO]
                .agg(cantidad="size", media="mean", mediana="median")
                .sort_values("media", ascending=False))
por_distrito.style.format({"media": "{:,.0f}", "mediana": "{:,.0f}"})

In [ ]:
orden = por_distrito.index
plt.figure(figsize=(12, 5))
sns.boxplot(data=df, x="distrito", y=OBJETIVO, order=orden, showfliers=False)
plt.xticks(rotation=45, ha="right"); plt.title("Precio por distrito (sin outliers)")
plt.tight_layout()

In [ ]:
# Numérica vs. objetivo: correlación
corr = df[numericas + [OBJETIVO]].corr(numeric_only=True)
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0)
plt.title("Matriz de correlación")

> 🕵️ **Misterio:** el área construida es siempre ~95% del área total, así que `area_m2` y `area_construida_m2` deberían tener una correlación ≈ 0.99. El heatmap muestra mucho menos.
> Además, `area_construida_m2` se correlaciona con el precio bastante más que `area_m2`. **¿Qué está "rompiendo" la correlación de `area_m2`?**
> Pista: mira las filas con el `area_m2` más grande. La correlación de Pearson es muy sensible a los **outliers** (módulo 2).
> Una vez limpias, las dos áreas serán redundantes: dicen lo mismo y sobra una (clase 15).

In [ ]:
# Comprueba la pista
df.nlargest(8, "area_m2")[["tipo_inmueble", "area_m2", "area_construida_m2", "precio_usd"]]

### ✏️ Tu turno
Haz el bivariante de `estado` contra `precio_usd`. ¿Tiene sentido el orden (Nuevo > Bueno > Regular > Remodelar)?

In [ ]:
# Tu código aquí


## Paso 4 (adelanto) · Datos faltantes ⭐
Es el tema de las clases 02 a 05.

In [ ]:
faltantes = df.isna().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)
pd.DataFrame({"faltantes": faltantes, "porcentaje": (faltantes / len(df) * 100).round(2)})

In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(df.isna().T, cbar=False, cmap="Greys")
plt.title("Mapa de faltantes (negro = NaN)"); plt.xlabel("fila")

### ✏️ Tu turno: piensa como científico de datos
- `piso` tiene muchísimos NaN. Revisa `df.groupby("tipo_inmueble")["piso"].apply(lambda s: s.isna().mean())`. ¿Es un error, o tiene sentido que falte?
- ¿En qué distritos falta más `distancia_centro_km`?

Hay faltantes **al azar** y faltantes que **dependen de otra variable**. Esa diferencia (MCAR / MAR / MNAR) es justo lo que verás en la clase 02.

In [ ]:
# Tu código aquí


## ✅ Siguiente paso
1. Abre `ejercicio.py` e implementa las 6 funciones (reutilizando lo que hiciste aquí).
2. Desde la raíz del repo:
```bash
make evaluar CLASE=cursos/nivelacion/n1-python-data-science/m1-eda-faltantes/clase-01-eda
```
3. Con nota ≥ 70, en Claude Code: `/evaluar cursos/nivelacion/n1-python-data-science/m1-eda-faltantes/clase-01-eda`